### Блокнот чисто для тестов со Spark. **Dev-only**
Происходит преобразование сырых данных с ingestion layer в очищенные дедуплицированные строки с преобразованием типов.


In [1]:
try: spark.stop()
except: pass

from pyspark.sql import SparkSession

PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

spark = (SparkSession.builder
    .appName("dev")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-beebfc69-ce37-4029-9507-2498f46ac716;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#c

In [2]:
df = spark.read.parquet("s3a://crypto-lake/bronze/crypto.tickers/")
bronze_schema = df.schema  # сериализованная схема
print(bronze_schema)

StructType([StructField('raw_json', StringType(), True), StructField('topic', StringType(), True), StructField('partition', IntegerType(), True), StructField('offset', LongType(), True), StructField('kafka_ts', TimestampType(), True), StructField('ingestion_ts', TimestampType(), False), StructField('year', IntegerType(), True), StructField('month', IntegerType(), True), StructField('day', IntegerType(), True), StructField('hour', IntegerType(), True)])


In [14]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, LongType, TimestampType

TOPIC = "crypto.tickers"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/bronze/{TOPIC}"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/silver/{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"
SILVER_PATH = f"s3a://crypto-lake/silver/{TOPIC}"

# схема вложенной структуры полезной нагрузки
data_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("lastPrice", StringType(), True),
    StructField("highPrice24h", StringType(), True),
    StructField("lowPrice24h", StringType(), True),
    StructField("prevPrice24h", StringType(), True),
    StructField("volume24h", StringType(), True),
    StructField("turnover24h", StringType(), True),
    StructField("price24hPcnt", StringType(), True),
    StructField("usdIndexPrice", StringType(), True),
])

payload_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("type", StringType(), True),
    StructField("cs", LongType(), True),
    StructField("topic", StringType(), True),
    StructField("data", data_schema, True),
])


PRICE_T = DecimalType(18, 8)
VOL_T = DecimalType(22, 4)
PCT_T = DecimalType(10, 6)

source = (spark.readStream
    .format("parquet")
    .schema(bronze_schema) 
    .option("maxFilesPerTrigger", 100)
    .load(BRONZE_PATH))

clean = (source
    .withColumn("p", F.from_json(F.col("raw_json"), payload_schema))
    .select(
        F.col("ingestion_ts"),
        F.col("kafka_ts"),
        F.timestamp_millis(F.col("p.ts")).alias("event_time"),
        F.col("p.type").alias("type"),
        F.col("p.cs").alias("cs"),
        F.col("p.data.symbol").alias("symbol"),
        F.col("p.data.lastPrice").cast(PRICE_T).alias("last_price"),
        F.col("p.data.highPrice24h").cast(PRICE_T).alias("high_price_24h"),
        F.col("p.data.lowPrice24h").cast(PRICE_T).alias("low_price_24h"),
        F.col("p.data.prevPrice24h").cast(PRICE_T).alias("prev_price_24h"),
        F.col("p.data.volume24h").cast(VOL_T).alias("volume_24h"),
        F.col("p.data.turnover24h").cast(VOL_T).alias("turnover_24h"),
        F.col("p.data.price24hPcnt").cast(PCT_T).alias("price_24h_pct"),
        F.col("p.data.usdIndexPrice").cast(PRICE_T).alias("usd_index_price"),
    )).dropDuplicates(["event_time", "cs"]).where(~F.isnull("usd_index_price"))\
        .withColumn("year",  F.year("event_time"))\
        .withColumn("month", F.month("event_time"))\
        .withColumn("day",   F.dayofmonth("event_time"))

'''write = (clean.writeStream
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="60 seconds")
    .start())
write.awaitTermination()'''

write = (clean.writeStream
    .format("parquet")
    .option("path", SILVER_PATH)
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .partitionBy("year", "month", "day")
    .trigger(processingTime="60 seconds")
    .start())

write.awaitTermination()

-------------------------------------------
Batch: 23
-------------------------------------------
+-----------------------+-----------------------+-----------------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |event_time             |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-----------------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-01 07:42:28.754|2026-05-01 07:41:57.331|2026-05-01 07:41:57.379|snapshot|106560485701|BTCUSDT|77037.60000000|77460.70000000|75879.40000000|76058.40000000|6130.5227 |468976310.4471|0.012900     |77006.16622300 |
|2026-05-01 07:32:19.9

-------------------------------------------
Batch: 35
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 09:38:19.267|2026-05-02 09:37:22.167|1777714643829|snapshot|106618089587|BTCUSDT|78396.30000000|78938.20000000|77154.40000000|77303.40000000|6584.2501 |515036262.9215|0.014100     |78386.09152200 |
|2026-05-02 09:37:11.812|2026-05-02 09:36:40.717|1777714602379

-------------------------------------------
Batch: 37
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 10:10:59.842|2026-05-02 10:10:51.254|1777716652979|snapshot|106618926803|BTCUSDT|78300.40000000|78938.20000000|77154.40000000|77213.50000000|6540.1093 |511688503.9008|0.014100     |78299.26887700 |
|2026-05-02 10:04:15.983|2026-05-02 10:03:08.419|1777716190130

ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

...

-------------------------------------------
Batch: 56
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 15:16:19.605|2026-05-02 15:16:16.917|1777734979379|snapshot|106627744700|BTCUSDT|78426.80000000|78753.50000000|77755.30000000|78516.10000000|4171.0928 |326650838.5194|-0.001100    |78432.36502100 |
|2026-05-02 15:16:01.491|2026-05-02 15:15:56.918|1777734959379

-------------------------------------------
Batch: 58
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 15:16:11.858|2026-05-02 15:16:06.918|1777734969379|snapshot|106627737381|BTCUSDT|78440.20000000|78753.50000000|77755.30000000|78516.10000000|4170.4143 |326597612.2079|-0.001000    |78434.16317700 |
|2026-05-02 15:16:11.858|2026-05-02 15:16:07.922|1777734970379

-------------------------------------------
Batch: 43
-------------------------------------------
+-----------------------+-----------------------+-----------------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |event_time             |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-----------------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 13:01:59.481|2026-05-02 13:00:59.183|2026-05-02 13:01:01.379|snapshot|106623663792|BTCUSDT|78231.90000000|78938.20000000|77755.30000000|78043.70000000|5691.5122 |445898998.5074|0.002400     |78225.03881900 |
|2026-05-02 12:48:27.6

-------------------------------------------
Batch: 55
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 15:17:31.973|2026-05-02 15:17:23.915|1777735046379|snapshot|106627794933|BTCUSDT|78389.90000000|78753.50000000|77755.30000000|78484.10000000|4145.3303 |324628004.4727|-0.001200    |78382.43259000 |
|2026-05-02 15:17:11.361|2026-05-02 15:17:08.964|1777735031429

-------------------------------------------
Batch: 57
-------------------------------------------
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|ingestion_ts           |kafka_ts               |ts           |type    |cs          |symbol |last_price    |high_price_24h|low_price_24h |prev_price_24h|volume_24h|turnover_24h  |price_24h_pct|usd_index_price|
+-----------------------+-----------------------+-------------+--------+------------+-------+--------------+--------------+--------------+--------------+----------+--------------+-------------+---------------+
|2026-05-02 15:17:31.973|2026-05-02 15:17:23.915|1777735046379|snapshot|106627794933|BTCUSDT|78389.90000000|78753.50000000|77755.30000000|78484.10000000|4145.3303 |324628004.4727|-0.001200    |78382.43259000 |
|2026-05-02 15:17:11.361|2026-05-02 15:17:08.964|1777735031429